# 11 — Secure RAG: tenant isolation and untrusted content

**Level:** Advanced · **Estimated time:** 90 minutes · **Scenario:** NovaTech Enterprise Knowledge Assistant

An Acme support user asks about checkout latency. The corpus also contains a restricted Globex renewal plan and an imported vendor note that attempts an indirect prompt injection. Build and test the boundary that decides what may reach retrieval, context, and an answer.


## Learning outcomes

You will enforce tenant and classification authorization **before** ranking, quarantine retrieved prompt-injection content, label retained documents as untrusted data, produce an auditable trace, and test no-leak/no-answer behavior.

The code is deterministic and intentionally small. Its purpose is to teach the control boundaries around a model, not to claim that a keyword detector alone solves prompt injection.


## 1. Threat model: where the system must decide

```text
principal + request
        ↓
authorize tenant / role / classification
   ├─ denied → exclude before retrieval and trace denial
   └─ allowed → inspect content as UNTRUSTED DATA
                  ├─ injection signal → quarantine + trace
                  └─ safe candidate → rank + bounded context
                                      ↓
                           answer from evidence or abstain
```

Two rules are non-negotiable:

1. A model cannot be trusted to enforce authorization after it has seen sensitive content.
2. Retrieved documents, tool output, and OCR text are data—not higher-priority instructions.


In [ ]:
from pathlib import Path
from src.rag_core.security_rag import (
    Principal, SecureEvidence, SecurityTrace, build_untrusted_context,
    decide_response, secure_retrieve,
)

ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path('../..')
security_dir = ROOT / 'data/enterprise/security'
acme = Principal('maya', 'acme', frozenset({'internal'}))
corpus = [
    SecureEvidence('acme-runbook', 'acme', 'internal', (security_dir / 'acme-checkout-runbook.md').read_text(), 'acme-checkout-runbook.md'),
    SecureEvidence('globex-plan', 'globex', 'internal', (security_dir / 'globex-renewal-plan.md').read_text(), 'globex-renewal-plan.md'),
    SecureEvidence('vendor-note', 'acme', 'internal', (security_dir / 'vendor-note.md').read_text(), 'vendor-note.md'),
]
print([(item.evidence_id, item.tenant_id, item.source) for item in corpus])


## 2. Authorization belongs before retrieval

Filtering after retrieval is too late: an unauthorized chunk can influence rank, a model context, caches, traces, or an intermediate agent decision. The retriever should only receive an authorized corpus. In production, the policy engine normally receives signed identity claims, tenant context, document labels, and purpose-of-use—not user-provided text alone.


In [ ]:
trace = SecurityTrace()
selected = secure_retrieve(acme, 'checkout incident approval', corpus, trace=trace)
print('Selected:', [item.evidence_id for item in selected])
print('Trace:', *trace.events, sep='\n- ')
assert 'globex-plan' not in {item.evidence_id for item in selected}


## 3. Indirect prompt injection crosses a trust boundary

An attacker can place instructions in a document, webpage, ticket, spreadsheet, image, or tool response. The system must not promote them into trusted instructions. The lab’s imported vendor note says to ignore instructions and bypass authorization; the detector quarantines it and keeps a trace event.

Real systems should layer provenance controls, ingestion review, content classification, instruction/data separation, least-privilege tools, output checks, monitoring, and a safe escalation path. Pattern matching is only a testable teaching control.


In [ ]:
injection_trace = SecurityTrace()
injection_query = 'authorization records'
injection_selected = secure_retrieve(acme, injection_query, corpus, trace=injection_trace)
print('Selected:', [item.evidence_id for item in injection_selected])
print('Trace:', *injection_trace.events, sep='\n- ')
assert any(event.startswith('quarantined:vendor-note') for event in injection_trace.events)


## 4. Context has an explicit trust label

Retained evidence is wrapped as `untrusted_document`. That label is an interface contract for the prompt builder and reviewer: it makes clear that the content can inform an answer but cannot override application policy, system instructions, permissions, budgets, or tool approval.


In [ ]:
context = build_untrusted_context(selected)
print(context)
assert 'globex-plan' not in context
assert 'vendor-note' not in context


## 5. Safe failure: abstain when no authorized evidence remains

An Acme user asking about Globex’s renewal discounts must not receive a softened or partial answer. The correct outcome is a safe no-answer state with a trace that distinguishes “no authorized evidence” from “the system has no data.” Avoid confirming the existence, title, or contents of inaccessible documents.


In [ ]:
denial_trace = SecurityTrace()
denied = secure_retrieve(acme, 'Globex renewal discount', corpus, trace=denial_trace)
decision = decide_response(denied, denial_trace)
print(decision)
print(*denial_trace.events, sep='\n- ')
assert not decision.allowed


## 6. Deliberate failure: post-retrieval filtering

Imagine a flawed implementation that ranks the whole corpus and removes foreign results just before rendering. It can still leak through model context, logs, ranking explanations, cache keys, or an agent’s next action. The fix is architectural: scope the corpus before retrieval and test that excluded IDs never appear in any trace field that feeds the model.


In [ ]:
# A testable safety invariant for every request.
assert all(item.tenant_id == acme.tenant_id for item in selected)
assert all('globex-plan' not in event for event in trace.events if event.startswith('retrieval:selected'))


## 7. Experiment: extend the policy

Try one change at a time:

1. Add a `restricted` document to Acme and show that a principal without the role cannot retrieve it.
2. Add an allowlisted source registry; quarantine documents from an unknown importer before they are indexed.
3. Add a tool request such as `restart_checkout`; prove that authorization and human approval remain application checks outside the retrieved content.
4. Add a prompt-injection test fixture in another modality, such as an OCR region or CSV comment field.

**Success criterion:** a test fails whenever foreign, restricted, or quarantined evidence can reach model context.


## 8. Production checklist

- Verify identity and enforce tenant, role, classification, and purpose-of-use before retrieval.
- Treat documents, retrieved text, OCR, and tool results as untrusted inputs.
- Preserve retrieval candidates, authorization decisions, quarantine reasons, and final evidence IDs in a privacy-conscious trace.
- Do not reveal whether a forbidden document exists; use a policy-safe response.
- Keep tool permission, approvals, rate limits, budgets, and side effects outside the model prompt.
- Run adversarial regression tests for cross-tenant retrieval, indirect prompt injection, data exfiltration requests, and privilege escalation.

## Checkpoint

1. Why is filtering after retrieval unsafe even if the UI hides the document?
2. What information should an injection-quarantine trace contain?
3. Why should a system label retrieved content as untrusted data?
4. What should the assistant do when no authorized evidence supports an answer?

### References

- [OWASP Top 10 for LLM Applications](https://genai.owasp.org/llm-top-10/)
- [OWASP prompt injection prevention cheat sheet](https://cheatsheetseries.owasp.org/cheatsheets/LLM_Prompt_Injection_Prevention_Cheat_Sheet.html)
- [NIST AI Risk Management Framework](https://www.nist.gov/itl/ai-risk-management-framework)
- [RAG evaluation lab: security and permissions](../evaluation/08_security_and_permissions.ipynb)
